# 2019 melt season: Vernagtferner

MODIS NIR and Sentinel-1 / Sentinel-2 glacier snow cover fractions. Run the cells in order after installing the dependencies in `README.md`. Google Earth Engine access is required.

In [ ]:
import ee
import matplotlib.pyplot as plt
from glacier_timeseries import MODIS_meanNIR, SCAF_S2, SCAF_S1

# Authenticate once on this machine if needed:
# ee.Authenticate()
ee.Initialize(project="YOUR_EARTH_ENGINE_PROJECT")

## Glacier and period

Vernagtferner uses the GLIMS identifier found in the source notebooks. The most recent outline by `src_date` is selected. Change the ID or supply another EE geometry for a different glacier. Dates are inclusive.

In [ ]:
start_date = "2019-06-01"
end_date = "2019-09-30"
glacier = (ee.FeatureCollection("GLIMS/current")
           .filter(ee.Filter.eq("glac_id", "G010823E46875N"))
           .sort("src_date", False)
           .first())
glacier_outline = glacier.geometry()

## Build the three time series

S2 thresholds use unscaled B8 digital numbers. The Vernagtferner example uses the notebook’s 3500–5000 range and 4500 fallback. SLA correction is enabled by default.

In [ ]:
modis_nir = MODIS_meanNIR(start_date, end_date, glacier_outline)
s2_scaf = SCAF_S2(start_date, end_date, glacier_outline,
                  th_min=3500, th_max=5000, th_fixed=4500)
s1_scaf = SCAF_S1(start_date, end_date, glacier_outline)
print(f"MODIS: {len(modis_nir)} dates; S2: {len(s2_scaf)} scenes; S1: {len(s1_scaf)} scenes")

## MODIS mean NIR

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(modis_nir.index, modis_nir.values, marker=".", linestyle="-",
        linewidth=1, label="MODIS mean NIR")
ax.set(xlabel="Acquisition date", ylabel="Mean NIR (native MOD09GQ units)",
       title="Vernagtferner · MODIS NIR · 2019")
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()

## Sentinel snow cover fractions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(s2_scaf.index, s2_scaf.values, "o-", label="Sentinel-2", markersize=4)
ax.plot(s1_scaf.index, s1_scaf.values, "s-", label="Sentinel-1", markersize=4)
ax.set(xlabel="Acquisition date", ylabel="Snow cover fraction (%)",
       ylim=(0, 100), title="Vernagtferner · SLA-corrected snow cover · 2019")
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()